In [1]:
import pandas as pd
import numpy as np

In [2]:
hospital_df = pd.read_excel(r"C:\Users\manda\Healthcare_analytics_project\data\hospital_final_dataset.xlsx")

mapping_df = pd.read_excel(
    "../data/disease_season_mapping.xlsx",
    sheet_name="Disease_Profile"
)

weights_df = pd.read_excel(
    "../data/disease_season_mapping.xlsx",
    sheet_name="Seasonal_Weights"
)

In [3]:
print("Hospital Dataset Shape:", hospital_df.shape)
print("Disease Mapping Shape:", mapping_df.shape)
print("Seasonal Weights Shape:", weights_df.shape)

Hospital Dataset Shape: (7999, 72)
Disease Mapping Shape: (26, 4)
Seasonal Weights Shape: (4, 13)


In [6]:
hospital_df = hospital_df.merge(
    mapping_df[
        [
            "APR MDC Description",
            "Season Profile",
            "Seasonal Pattern"
        ]
    ],
    on="APR MDC Description",
    how="left"
)

In [7]:
hospital_df[
    [
        "APR MDC Description",
        "Season Profile",
        "Seasonal Pattern"
    ]
].head(10)

,APR MDC Description,Season Profile,Seasonal Pattern
0,INFECTIOUS AND PARASITIC DISEASES (SYSTEMIC OR...,Monsoon,Monsoon Peak
1,DISEASES AND DISORDERS OF THE NERVOUS SYSTEM,Uniform,Uniform Distribution
2,"INJURIES, POISONINGS AND TOXIC EFFECTS OF DRUGS",Uniform,Uniform Distribution
3,DISEASES AND DISORDERS OF THE CIRCULATORY SYSTEM,Uniform,Uniform Distribution
4,NEWBORNS AND OTHER NEONATES WITH CONDITIONS OR...,Uniform,Uniform Distribution
5,BURNS,Uniform,Uniform Distribution
6,INFECTIOUS AND PARASITIC DISEASES (SYSTEMIC OR...,Monsoon,Monsoon Peak
7,"PREGNANCY, CHILDBIRTH AND THE PUERPERIUM",Uniform,Uniform Distribution
8,NEWBORNS AND OTHER NEONATES WITH CONDITIONS OR...,Uniform,Uniform Distribution
9,"PREGNANCY, CHILDBIRTH AND THE PUERPERIUM",Uniform,Uniform Distribution


In [8]:
hospital_df["Season Profile"].isna().sum()

np.int64(0)

In [9]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

hospital_df["LOS_Norm"] = scaler.fit_transform(
    hospital_df[["Length of Stay"]]
)

hospital_df["Severity_Norm"] = scaler.fit_transform(
    hospital_df[["Severity_Score"]]
)

hospital_df["Mortality_Norm"] = scaler.fit_transform(
    hospital_df[["Mortality_Risk_Score"]]
)

In [10]:
hospital_df["Operational_Score"] = (
    0.35 * hospital_df["Severity_Norm"] +
    0.30 * hospital_df["LOS_Norm"] +
    0.20 * hospital_df["Emergency_Flag"] +
    0.15 * hospital_df["Mortality_Norm"]
)

In [11]:
hospital_df[
    [
        "Length of Stay",
        "Severity_Score",
        "Mortality_Risk_Score",
        "Emergency_Flag",
        "Operational_Score"
    ]
].head(10)

,Length of Stay,Severity_Score,Mortality_Risk_Score,Emergency_Flag,Operational_Score
0,4,4,4,0,0.507563
1,1,2,1,0,0.212500
2,5,2,2,1,0.460084
3,7,3,3,0,0.390126
4,3,1,1,0,0.130042
5,5,2,1,1,0.422584
6,1,4,4,1,0.700000
7,2,1,1,1,0.327521
8,2,1,1,0,0.127521
9,2,1,1,1,0.327521


In [12]:
hospital_df["Operational_Score"].describe()

count    7999.000000
mean        0.366217
std         0.173956
min         0.000000
25%         0.215021
50%         0.355147
75%         0.480147
max         1.000000
Name: Operational_Score, dtype: float64

In [13]:
import pandas as pd

seasonal_weights = pd.read_excel(
    r"C:\Users\manda\Healthcare_analytics_project\data\disease_season_mapping.xlsx",
    sheet_name="Seasonal_Weights"
)

seasonal_weights

,Profile,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,Uniform,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.37
1,Winter,14.00,13.00,10.00,8.00,6.00,5.00,5.00,5.00,6.00,8.00,10.00,10.00
2,Summer,6.00,6.00,7.00,9.00,11.00,12.00,11.00,10.00,9.00,7.00,6.00,6.00
3,Monsoon,5.00,5.00,6.00,7.00,8.00,11.00,14.00,14.00,12.00,9.00,5.00,4.00


In [14]:
weights_lookup = seasonal_weights.set_index("Profile").to_dict(orient="index")

weights_lookup

{'Uniform': {'Jan': 8.33,
  'Feb': 8.33,
  'Mar': 8.33,
  'Apr': 8.33,
  'May': 8.33,
  'Jun': 8.33,
  'Jul': 8.33,
  'Aug': 8.33,
  'Sep': 8.33,
  'Oct': 8.33,
  'Nov': 8.33,
  'Dec': 8.37},
 'Winter': {'Jan': 14.0,
  'Feb': 13.0,
  'Mar': 10.0,
  'Apr': 8.0,
  'May': 6.0,
  'Jun': 5.0,
  'Jul': 5.0,
  'Aug': 5.0,
  'Sep': 6.0,
  'Oct': 8.0,
  'Nov': 10.0,
  'Dec': 10.0},
 'Summer': {'Jan': 6.0,
  'Feb': 6.0,
  'Mar': 7.0,
  'Apr': 9.0,
  'May': 11.0,
  'Jun': 12.0,
  'Jul': 11.0,
  'Aug': 10.0,
  'Sep': 9.0,
  'Oct': 7.0,
  'Nov': 6.0,
  'Dec': 6.0},
 'Monsoon': {'Jan': 5.0,
  'Feb': 5.0,
  'Mar': 6.0,
  'Apr': 7.0,
  'May': 8.0,
  'Jun': 11.0,
  'Jul': 14.0,
  'Aug': 14.0,
  'Sep': 12.0,
  'Oct': 9.0,
  'Nov': 5.0,
  'Dec': 4.0}}

In [17]:
hospital_df = hospital_df.merge(
    seasonal_weights,
    left_on="Season Profile",
    right_on="Profile",
    how="left"
)

hospital_df.head()

,Hospital Service Area,Hospital County,Operating Certificate Number,Permanent Facility Id,Facility Name,Age Group,Zip Code - 3 digits,Gender,Race,Ethnicity,...,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,New York City,Manhattan,7002054,1458,New York-Presbyterian Hospital - New York Weil...,70 or Older,OOS,M,Other Race,Unknown,...,6.00,7.00,8.00,11.00,14.00,14.00,12.00,9.00,5.00,4.00
1,New York City,Manhattan,7002054,1458,New York-Presbyterian Hospital - New York Weil...,0 to 17,114,F,Other Race,Spanish/Hispanic,...,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.37
2,New York City,Manhattan,7002054,1458,New York-Presbyterian Hospital - New York Weil...,30 to 49,100,M,Black/African American,Spanish/Hispanic,...,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.37
3,New York City,Manhattan,7002054,1458,New York-Presbyterian Hospital - New York Weil...,70 or Older,103,F,Other Race,Unknown,...,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.37
4,New York City,Manhattan,7002054,1458,New York-Presbyterian Hospital - New York Weil...,0 to 17,100,M,White,Not Span/Hispanic,...,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.37


In [18]:
hospital_df[
    [
        "Season Profile",
        "Jan",
        "Feb",
        "Mar",
        "Apr"
    ]
].head()

,Season Profile,Jan,Feb,Mar,Apr
0,Monsoon,5.00,5.00,6.00,7.00
1,Uniform,8.33,8.33,8.33,8.33
2,Uniform,8.33,8.33,8.33,8.33
3,Uniform,8.33,8.33,8.33,8.33
4,Uniform,8.33,8.33,8.33,8.33


In [22]:
months = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

for month in months:
    hospital_df[f"{month}_Contribution"] = (
        hospital_df["Operational_Score"] *
        hospital_df[month] / 100
    )

In [19]:
hospital_df[
    [
        "Season Profile",
        "Operational_Score",
        "Jan_Contribution",
        "Feb_Contribution",
        "Mar_Contribution"
    ]
].head()


,Season Profile,Operational_Score,Jan_Contribution,Feb_Contribution,Mar_Contribution
0,Monsoon,0.507563,0.025378,0.025378,0.030454
1,Uniform,0.212500,0.017701,0.017701,0.017701
2,Uniform,0.460084,0.038325,0.038325,0.038325
3,Uniform,0.390126,0.032497,0.032497,0.032497
4,Uniform,0.130042,0.010833,0.010833,0.010833


In [23]:
contribution_cols = [f"{month}_Contribution" for month in months]

hospital_df["Contribution_Total"] = hospital_df[contribution_cols].sum(axis=1)

hospital_df[
    [
        "Operational_Score",
        "Contribution_Total"
    ]
].head()

,Operational_Score,Contribution_Total
0,0.507563,0.507563
1,0.212500,0.212500
2,0.460084,0.460084
3,0.390126,0.390126
4,0.130042,0.130042


In [24]:
months = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

monthly_summary = []

for month in months:

    contribution_col = f"{month}_Contribution"

    monthly_summary.append({
        "Month": month,

        "Estimated_Admissions":
            hospital_df[contribution_col].sum(),

        "Average_Operational_Score":
            hospital_df["Operational_Score"].mean(),

        "Average_Length_of_Stay":
            hospital_df["Length of Stay"].mean(),

        "Emergency_Percentage":
            hospital_df["Emergency_Flag"].mean() * 100,

        "Average_Severity":
            hospital_df["Severity_Score"].mean(),

        "Average_Mortality_Risk":
            hospital_df["Mortality_Risk_Score"].mean()
    })

monthly_trends = pd.DataFrame(monthly_summary)

monthly_trends

,Month,Estimated_Admissions,Average_Operational_Score,Average_Length_of_Stay,Emergency_Percentage,Average_Severity,Average_Mortality_Risk
0,Jan,244.756315,0.366217,5.734967,57.032129,2.018752,1.695337
1,Feb,242.030541,0.366217,5.734967,57.032129,2.018752,1.695337
2,Mar,239.521959,0.366217,5.734967,57.032129,2.018752,1.695337
3,Apr,243.900655,0.366217,5.734967,57.032129,2.018752,1.695337
4,May,248.279350,0.366217,5.734967,57.032129,2.018752,1.695337
5,Jun,254.236791,0.366217,5.734967,57.032129,2.018752,1.695337
6,Jul,254.597000,0.366217,5.734967,57.032129,2.018752,1.695337
7,Aug,250.435497,0.366217,5.734967,57.032129,2.018752,1.695337
8,Sep,245.985293,0.366217,5.734967,57.032129,2.018752,1.695337
9,Oct,238.592123,0.366217,5.734967,57.032129,2.018752,1.695337


In [25]:
monthly_trends = monthly_trends.round({
    "Estimated_Admissions": 2,
    "Average_Operational_Score": 3,
    "Average_Length_of_Stay": 2,
    "Emergency_Percentage": 2,
    "Average_Severity": 2,
    "Average_Mortality_Risk": 2
})

monthly_trends

,Month,Estimated_Admissions,Average_Operational_Score,Average_Length_of_Stay,Emergency_Percentage,Average_Severity,Average_Mortality_Risk
0,Jan,244.76,0.366,5.73,57.03,2.02,1.7
1,Feb,242.03,0.366,5.73,57.03,2.02,1.7
2,Mar,239.52,0.366,5.73,57.03,2.02,1.7
3,Apr,243.90,0.366,5.73,57.03,2.02,1.7
4,May,248.28,0.366,5.73,57.03,2.02,1.7
5,Jun,254.24,0.366,5.73,57.03,2.02,1.7
6,Jul,254.60,0.366,5.73,57.03,2.02,1.7
7,Aug,250.44,0.366,5.73,57.03,2.02,1.7
8,Sep,245.99,0.366,5.73,57.03,2.02,1.7
9,Oct,238.59,0.366,5.73,57.03,2.02,1.7


In [26]:
output_file = "../data/monthly_operational_trends.xlsx"

monthly_trends.to_excel(output_file, index=False)

print(f"✅ File created successfully: {output_file}")

✅ File created successfully: ../data/monthly_operational_trends.xlsx


In [27]:
months = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

monthly_summary = []

for month in months:

    contribution_col = f"{month}_Contribution"

    total_contribution = hospital_df[contribution_col].sum()

    monthly_summary.append({

        "Month": month,

        "Estimated_Admissions": total_contribution,

        "Average_Operational_Score":
            (hospital_df["Operational_Score"] * hospital_df[contribution_col]).sum() / total_contribution,

        "Average_Length_of_Stay":
            (hospital_df["Length of Stay"] * hospital_df[contribution_col]).sum() / total_contribution,

        "Emergency_Percentage":
            (hospital_df["Emergency_Flag"] * hospital_df[contribution_col]).sum() / total_contribution * 100,

        "Average_Severity":
            (hospital_df["Severity_Score"] * hospital_df[contribution_col]).sum() / total_contribution,

        "Average_Mortality_Risk":
            (hospital_df["Mortality_Risk_Score"] * hospital_df[contribution_col]).sum() / total_contribution

    })

monthly_trends = pd.DataFrame(monthly_summary)

In [28]:
monthly_trends = monthly_trends.round({
    "Estimated_Admissions": 2,
    "Average_Operational_Score": 3,
    "Average_Length_of_Stay": 2,
    "Emergency_Percentage": 2,
    "Average_Severity": 2,
    "Average_Mortality_Risk": 2
})

monthly_trends


,Month,Estimated_Admissions,Average_Operational_Score,Average_Length_of_Stay,Emergency_Percentage,Average_Severity,Average_Mortality_Risk
0,Jan,244.76,0.450,7.79,73.51,2.39,2.05
1,Feb,242.03,0.449,7.77,73.38,2.39,2.04
2,Mar,239.52,0.448,7.75,73.27,2.38,2.03
3,Apr,243.90,0.447,7.74,73.46,2.37,2.02
4,May,248.28,0.447,7.73,73.64,2.37,2.02
5,Jun,254.24,0.450,7.81,74.00,2.38,2.04
6,Jul,254.60,0.452,7.92,74.16,2.40,2.05
7,Aug,250.44,0.452,7.92,74.01,2.39,2.05
8,Sep,245.99,0.450,7.88,73.76,2.39,2.04
9,Oct,238.59,0.449,7.82,73.35,2.38,2.03


In [29]:
monthly_trends.to_excel(
    r"C:\Users\manda\Healthcare_analytics_project\data\monthly_operational_trends.xlsx",
    index=False
)

print("✅ Updated monthly_operational_trends.xlsx created successfully!")

✅ Updated monthly_operational_trends.xlsx created successfully!


In [30]:
hospital_df.to_excel(
    r"C:\Users\manda\Healthcare_analytics_project\data\hospital_final_dataset.xlsx",
    index=False
)

print("✅ Updated hospital_final_dataset.xlsx saved successfully.")

✅ Updated hospital_final_dataset.xlsx saved successfully.


In [31]:
print(hospital_df.columns.tolist())

['Hospital Service Area', 'Hospital County', 'Operating Certificate Number', 'Permanent Facility Id', 'Facility Name', 'Age Group', 'Zip Code - 3 digits', 'Gender', 'Race', 'Ethnicity', 'Length of Stay', 'Type of Admission', 'Patient Disposition', 'Discharge Year', 'CCSR Diagnosis Code', 'CCSR Diagnosis Description', 'CCSR Procedure Code', 'CCSR Procedure Description', 'APR DRG Code', 'APR DRG Description', 'APR MDC Code', 'APR MDC Description', 'APR Severity of Illness Code', 'APR Severity of Illness Description', 'APR Risk of Mortality', 'APR Medical Surgical Description', 'Payment Typology 1', 'Payment Typology 2', 'Birth Weight', 'Emergency Department Indicator', 'Total Charges', 'Total Costs', 'Stay_Category', 'Cost_Per_Day', 'Charge_Per_Day', 'Emergency_Flag', 'Severity_Score', 'Mortality_Risk_Score', 'Readmission_Risk_Flag', 'Department_Efficiency_Score', 'Season Profile_x', 'Seasonal Pattern_x', 'LOS_Norm', 'Severity_Norm', 'Mortality_Norm', 'Operational_Score', 'Profile_x', 'J

In [32]:
hospital_df.to_excel("hospital_final_dataset.xlsx", index=False)

In [33]:
print(hospital_df[['Jan_x', 'Jan_y', 'Jan']].head())

   Jan_x  Jan_y   Jan
0   5.00   5.00  5.00
1   8.33   8.33  8.33
2   8.33   8.33  8.33
3   8.33   8.33  8.33
4   8.33   8.33  8.33


In [34]:
print(hospital_df[['Feb_x', 'Feb_y', 'Feb']].head())

   Feb_x  Feb_y   Feb
0   5.00   5.00  5.00
1   8.33   8.33  8.33
2   8.33   8.33  8.33
3   8.33   8.33  8.33
4   8.33   8.33  8.33


In [36]:
hospital_df = hospital_df.drop(columns=[
    'Jan_x','Feb_x','Mar_x','Apr_x','May_x','Jun_x','Jul_x','Aug_x','Sep_x','Oct_x','Nov_x','Dec_x',
    'Jan_y','Feb_y','Mar_y','Apr_y','May_y','Jun_y','Jul_y','Aug_y','Sep_y','Oct_y','Nov_y','Dec_y'
])

KeyError: "['Jan_x', 'Feb_x', 'Mar_x', 'Apr_x', 'May_x', 'Jun_x', 'Jul_x', 'Aug_x', 'Sep_x', 'Oct_x', 'Nov_x', 'Dec_x', 'Jan_y', 'Feb_y', 'Mar_y', 'Apr_y', 'May_y', 'Jun_y', 'Jul_y', 'Aug_y', 'Sep_y', 'Oct_y', 'Nov_y', 'Dec_y'] not found in axis"

In [40]:
hospital_df.to_excel(
    r"C:\Users\manda\Healthcare_analytics_project\data\hospital_final_dataset1.xlsx",
    index=False
)

In [42]:
hospital_df[['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']].head(10)

,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,5.00,5.00,6.00,7.00,8.00,11.00,14.00,14.00,12.00,9.00,5.00,4.00
1,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.37
2,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.37
3,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.37
4,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.37
5,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.37
6,5.00,5.00,6.00,7.00,8.00,11.00,14.00,14.00,12.00,9.00,5.00,4.00
7,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.37
8,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.37
9,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.37


In [43]:
hospital_df['Profile'].value_counts()

Profile
Uniform    6169
Summer     1005
Winter      565
Monsoon     260
Name: count, dtype: int64